# Rekenkamer in Parliamentary Discourse

This notebook traces how the Dutch Court of Audit (*Rekenkamer*) appears in parliamentary debates
from the 19th century to the present.  It uses sentence-level hits matched against LDA topic
distributions to measure *when*, *by whom*, and *about what* the Rekenkamer is discussed.

**Data sources**
- `hits.csv` — sentence matches linking Rekenkamer reports to parliamentary speeches
- LDA model (300 topics, 1972–2022, chunked Handelingen)
- `proc-hits.csv` — noun totals for relative-frequency normalisation
- `resources/cabinets.csv` — cabinet composition for coalition/opposition classification

In [ ]:
from helper import *
from paths import HITS_CSV, LDA_DIR, NOUN_TOTAL_CSV, CABINETS_CSV

## 1  Load data

Paths are resolved via `paths.py`.  Set `REKENKAMER_LDA_DIR` in your shell if the LDA
output lives somewhere other than `~/Downloads/lda/data-1972-2022-chunked/`.

In [ ]:
hits, dist, meta, ks, total_nouns, cab = load_all(
    hits_path=HITS_CSV,
    lda_dir=LDA_DIR,
    noun_total_path=NOUN_TOTAL_CSV,
    cabinet_path=CABINETS_CSV,
)

TOPIC_COLS = list(range(300))

# Restrict to lower house for most analyses
hits_commons = hits[hits["house"] == "commons"].copy()

# Add half-year period labels and flag dist rows that appear in hits
hits_commons, dist = add_periods(hits_commons, dist)

# Quarterly-averaged topic distributions (for corpus-level comparisons)
dist_averaged = dist.groupby(dist["date"].dt.to_period("Q"))[TOPIC_COLS].mean()

## 2  Long-run frequency of references

How many times did speakers mention the Court of Audit, and how did this change over time?
The **red area** shows summed relative frequency (normalised by total noun count); the **blue line**
shows entropy — a measure of how spread out the references are across days within each quarter.

High entropy + high frequency = the Rekenkamer is discussed routinely by many speakers on many days.  
Low entropy + high frequency = a concentrated spike (one big debate, one landmark report).

In [ ]:
daily = relative_frequency(hits_commons, total_nouns)
quarterly = quarterly_stats(daily, lowess_frac=0.05)

fig = plot_freq_entropy(quarterly, war_shade=True)
fig.savefig("../results/figs/freq_entropy.png", dpi=150, bbox_inches="tight")

## 3  Seasonal patterns by week of year

Does the Rekenkamer follow the parliamentary calendar?  Each row below is a 5-year bin;
each column is a week of the year.  Intensity = share of that bin's hits falling in that week.

The *Verantwoordingsdag* (third Wednesday of May) and the annual budget debates should show up here.

In [ ]:
fig = plot_weekly_heatmap(hits_commons, year_start=1940, period_size=5)
fig.savefig("../results/figs/weekly_heatmap.png", dpi=150, bbox_inches="tight")

## 4  Who raises the Rekenkamer — government or opposition?

Each speaker is classified as **government** (minister / state secretary), **coalition backbencher**,
or **opposition MP** based on which cabinet was in office on that date.

If the opposition is driving mentions, it suggests the Rekenkamer is used as an accountability
instrument.  If the government drives it, the Rekenkamer may be cited to *legitimise* policy.

In [ ]:
hits_classified = classify_coalition(hits_commons, cab)

fig = plot_coalition_area(hits_classified, year_start=1945)
fig.savefig("../results/figs/coalition_area.png", dpi=150, bbox_inches="tight")

## 5  Topic–hit partial correlations over time

For each quarter we compute the partial correlation between each LDA topic's daily proportion
and the number of Rekenkamer hits — *controlling for all other topics simultaneously*.

A positive partial correlation means: on days when the corpus talks more about topic *T*, there are
also more Rekenkamer references, even after accounting for the general thematic ebb and flow.

In [ ]:
dam, hits_day = build_dam(dist, hits_commons)

In [ ]:
res = pcorr_over_time(dam, hits_day, min_obs=30)
print(f"{len(res):,} topic-quarter observations")

In [ ]:
res["label"] = res["topic"].astype(str) + " " + res["topic"].map(ks)
resp = res.groupby(["period", "label"])["partial_corr"].mean().unstack().dropna()

### 5.1  Topics with a rising association with the Rekenkamer

Mann-Kendall slope on the EWM-smoothed partial-correlation time series —
a positive slope means the topic has become *more* associated with Rekenkamer mentions over time.

In [ ]:
trending_topics(resp, n=15, ewm_span=12)

### 5.2  Topic landscape

**x** = variance of partial correlation over time (does the link fluctuate a lot?),  
**y** = mean partial correlation (how persistently associated with the Rekenkamer?),  
**bubble size** = peak association,  
**colour** = overall prominence in the corpus (log₂-softmax z-score).

Topics in the top-right corner are both strongly *and consistently* associated with the Rekenkamer.

In [ ]:
scatter_df = topic_scatter_data(resp, dist_averaged)
fig = plot_topic_bubble(scatter_df, ks, top_n_labels=12)
fig.savefig("../results/figs/topic_bubble.png", dpi=150, bbox_inches="tight")

### 5.3  Deep-dive: individual topics

**Blue** = partial correlation of topic with Rekenkamer hits per quarter.  
**Red dashed** = overall prevalence of the topic in the corpus.

When the two lines diverge the Rekenkamer association is not merely proportional to how prominent
the topic is in parliament — it reflects something specific about *oversight* discourse.

Edit `TOPICS_OF_INTEREST` to explore other topic IDs (check `ks[t]` for the keyword list).

In [ ]:
TOPICS_OF_INTEREST = [236, 195, 141, 77, 261]

for t in TOPICS_OF_INTEREST:
    fig = plot_topic_timeseries(resp, dist_averaged, t, ks, ewm_span=20)
    fig.savefig(f"../results/figs/topic_{t}_timeseries.png", dpi=150, bbox_inches="tight")

### 5.4  Heatmap of selected topics over time

Rows = topics from `TOPICS_OF_INTEREST`.  Columns = quarters.  
Red = strong positive association with Rekenkamer mentions; blue = negative.

In [ ]:
fig = plot_topic_heatmap(resp, TOPICS_OF_INTEREST, ks, ewm_span=12)
fig.savefig("../results/figs/topic_heatmap.png", dpi=150, bbox_inches="tight")

## 6  Pairwise topic co-occurrence in the full corpus

Which topics consistently appear together in parliamentary debate?
The heatmap below shows how strongly a focal topic's neighbourhood shifts across decades.

In [ ]:
import seaborn as sns

FOCAL_TOPIC = 236        # financiën, begroting, verantwoording …
SCORE_THRESHOLD = 0.1

yearly_pairs = []
for y, ds in dist.groupby(dist["date"].dt.year):
    try:
        pc = topic_corr_matrix(ds[TOPIC_COLS])
        long = (
            pc.stack().rename("score").reset_index()
            .rename(columns={"level_0": "topic_i", "level_1": "topic_j"})
        )
        long = long[
            (long["topic_i"] != long["topic_j"]) &
            (long["score"] > SCORE_THRESHOLD)
        ]
        long["topic_i"] = long["topic_i"].map(ks)
        long["topic_j"] = long["topic_j"].map(ks)
        yearly_pairs.append(long.assign(year=y))
    except Exception:
        continue

pairs_df = pd.concat(yearly_pairs, ignore_index=True)

In [ ]:
focal_label = ks[FOCAL_TOPIC]
subset = (
    pairs_df[pairs_df["topic_i"] == focal_label]
    .groupby(["year", "topic_j"])["score"].mean()
    .unstack()
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(subset.T, ax=ax, cmap="Blues", cbar_kws={"shrink": 0.6})
ax.set_title(f"Topics co-occurring with: {focal_label[:60]}", loc="left")
fig.tight_layout()
fig.savefig(f"../results/figs/pairwise_{FOCAL_TOPIC}.png", dpi=150, bbox_inches="tight")